In [1]:
from pyspark.sql import SparkSession
import json
import os
from pyspark.sql import functions as sf
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "forge-commerce-user")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "forge-commerce-pass")
S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio:9000")
PREFIX = "products"
RAW_BUCKET = "raw"
CLEANED_BUCKET = "cleaned"
CURATED_BUCKET = "curated"
RAW_PATH = f"s3a://{RAW_BUCKET}/{PREFIX}/"
CLEANED_PATH = f"s3a://{CLEANED_BUCKET}/{PREFIX}/"
CURATED_PATH = f"s3a://{CURATED_BUCKET}/{PREFIX}/"

In [2]:
spark = (
        SparkSession.builder.appName("test_products")
        .master(os.environ.get("SPARK_MASTER", "spark://spark-master:7077"))
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Delta Lake configurations
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .getOrCreate()
    )

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 00:59:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
df_curated_result = spark.read.format("delta").load(CURATED_PATH)
# df_curated_result.printSchema()
# df_curated_result.orderBy("product_id", "created_at").select("product_id", "product_name", "created_at", "effective_from", "effective_to", "is_active").show(30, truncate=False)
df_curated_result.orderBy("product_id", "created_at").select("*").show(30, truncate=False)

root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- color: string (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- description: string (nullable = true)
 |-- dimensions: string (nullable = true)
 |-- inventory_level: long (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- is_discontinued: boolean (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- margin: double (nullable = true)
 |-- material: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_rating: double (nullable = true)
 |-- product_uuid: string (nullable = true)
 |-- return_policy_days: long (nullable = true)
 |-- review_count: long (nullable = true)
 |-- sku: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- supplier_country: string (nullable = true)
 |-- supplier_name: st

+-------------+-------------+-----+----------+-------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------------+---------+---------------+-------------------+------+--------+------+----------+---------------------------+--------------+------------------------------------+------------------+------------+-----------+---------------+----------------+--------------------+-----------+-------------+---------------+------+-------------+--------------+-----------------------------+---------------+----------------------+----------------+-----------------+--------------+---------------+-----------------+---------+--------+---------+----------+----------------+--------------+----------------+---------------+-------------------------+-----------------------+-------------------+-------------------+
|brand        |categor

In [7]:
spark.stop()